# Flight Delay Intelligence System — Feature Engineering
This notebook prepares the final feature set for model training by removing leakage columns, selecting relevant features, and encoding categorical variables.

In [1]:
import pandas as pd
import numpy as np

## 1. Load Data
Loading flights_processed.csv — 2,944,078 rows, output of preprocessing notebook.

In [2]:
df=pd.read_csv("data/processed/flights_processed.csv")

In [3]:
df.shape

(2944078, 28)

## 2. Drop Leakage Columns
Removing all columns that would not be available at prediction time (pre-departure):
- `DEP_DELAY`, `ARR_DELAY` — actual delay values, only known after flight
- `DEP_TIME`, `ARR_TIME` — actual times, only known after flight
- `WHEELS_OFF`, `WHEELS_ON`, `TAXI_OUT`, `TAXI_IN` — operational data, post-departure
- `ACTUAL_ELAPSED_TIME`, `AIR_TIME` — post-flight data
- `CRS_ELAPSED_TIME` — 0.98 correlation with DISTANCE, redundant

In [4]:
df=df.drop(columns=['DEP_DELAY','ARR_DELAY','CRS_ELAPSED_TIME','WHEELS_OFF','WHEELS_ON','TAXI_OUT','TAXI_IN','DEP_TIME','ARR_TIME','ACTUAL_ELAPSED_TIME','AIR_TIME'])

In [5]:
df.shape

(2944078, 17)

In [24]:
df.columns.tolist()

['CRS_DEP_TIME',
 'CRS_ARR_TIME',
 'DISTANCE',
 'MONTH',
 'DAY_OF_WEEK',
 'ARR_DEL15',
 'CARRIER_ENCODED',
 'ORIGIN_ENCODED',
 'DEST_ENCODED']

## 3. Drop Non-Predictive Columns
- `OP_CARRIER_FL_NUM` — flight number is an identifier, no predictive signal
- `YEAR` — sampled equally across years, won't generalize to future data
- `DAY` — day of month is too granular, no meaningful pattern

In [9]:
df=df.drop(columns=['OP_CARRIER_FL_NUM','DAY','YEAR'])

## 4. Drop Delay Cause Columns
`CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY` are post-event data — only populated after a delay has already occurred. Using them would be data leakage.

In [12]:
df=df.drop(columns=['CARRIER_DELAY','WEATHER_DELAY','NAS_DELAY','SECURITY_DELAY','LATE_AIRCRAFT_DELAY'])

In [13]:
df.shape

(2944078, 9)

In [14]:
df.columns.tolist()

['OP_CARRIER',
 'ORIGIN',
 'DEST',
 'CRS_DEP_TIME',
 'CRS_ARR_TIME',
 'DISTANCE',
 'MONTH',
 'DAY_OF_WEEK',
 'ARR_DEL15']

## 5. Target Encoding for Categorical Columns
`OP_CARRIER` (20 unique), `ORIGIN` (367 unique), `DEST` (367 unique) are string columns.

One-hot encoding would create 734+ new columns — too sparse. Instead using **target encoding** — replacing each category with its mean delay rate.

- `OP_CARRIER` → `CARRIER_ENCODED`
- `ORIGIN` → `ORIGIN_ENCODED`
- `DEST` → `DEST_ENCODED`

Note: In production, encoding is fit on training data only and applied to test data to prevent leakage. This will be handled properly during model training.

In [19]:
carrier_encoding = df.groupby('OP_CARRIER')['ARR_DEL15'].mean()
df['CARRIER_ENCODED'] = df['OP_CARRIER'].map(carrier_encoding)

In [20]:
origin_encoding = df.groupby('ORIGIN')['ARR_DEL15'].mean()
df['ORIGIN_ENCODED'] = df['ORIGIN'].map(origin_encoding)

dest_encoding = df.groupby('DEST')['ARR_DEL15'].mean()
df['DEST_ENCODED'] = df['DEST'].map(dest_encoding)

In [21]:
df = df.drop(columns=['OP_CARRIER', 'ORIGIN', 'DEST'])

In [22]:
df.shape

(2944078, 9)

In [23]:
df.columns.tolist()

['CRS_DEP_TIME',
 'CRS_ARR_TIME',
 'DISTANCE',
 'MONTH',
 'DAY_OF_WEEK',
 'ARR_DEL15',
 'CARRIER_ENCODED',
 'ORIGIN_ENCODED',
 'DEST_ENCODED']

## 6. Save Feature Engineered Dataset
Saving final feature set to `data/processed/flights_features.csv`.

In [26]:
df.to_csv("data/processed/flights_features.csv", index=False)

## 7. Feature Engineering Summary

| Feature | Type | Description |
|---|---|---|
| `CRS_DEP_TIME` | Numerical | Scheduled departure hour |
| `CRS_ARR_TIME` | Numerical | Scheduled arrival hour |
| `DISTANCE` | Numerical | Route distance in miles |
| `MONTH` | Numerical | Month of flight |
| `DAY_OF_WEEK` | Numerical | Day of week (0=Mon, 6=Sun) |
| `CARRIER_ENCODED` | Numerical | Airline target encoded delay rate |
| `ORIGIN_ENCODED` | Numerical | Origin airport target encoded delay rate |
| `DEST_ENCODED` | Numerical | Destination airport target encoded delay rate |
| `ARR_DEL15` | Target | 1 = delayed 15+ min, 0 = on time |
